# Módulo 9 · De MNE a Machine Learning

**Objetivo:** juntar todo en un **pipeline reproducible** que recorra varios ficheros de TUSZ y produzca
`X`, `y` y `groups` (paciente), entrenar un **baseline** con scikit-learn evaluado correctamente
(sin fuga de pacientes) y exportar los datos para PyTorch. Es el puente directo a la fase 3 del TFM.

```
files.csv ──► lista de ficheros ──► read_tusz_raw ─► preprocess ─► make_windows ─► X, y, groups
                                                                                     │
                                    baseline sklearn (band power + LogReg) ◄─────────┤
                                    tensores PyTorch (señal cruda 22×800) ◄──────────┘
```

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from tusz_utils import (read_tusz_raw, preprocess, make_windows, band_power, find_edf,
                        INVENTORY, TUSZ_ROOT, BANDS, TCP_NAMES, SALIDAS)

mne.set_log_level("ERROR")

## 9.1 Elegir ficheros con el inventario
Empezamos por **todas las crisis de ausencia de train** (tu tipo favorito y un problema pequeño
para iterar rápido). Nos quedamos con los ficheros que tengan los 22 canales TCP (montajes LE y AR).

In [ ]:
files = pd.read_csv(INVENTORY / "files.csv")
sel = files[(files.split == "train") & files.labels.fillna("").str.contains("absz")
            & files.montage.isin(["01_tcp_ar", "02_tcp_le"])]
print(len(sel), "ficheros de", sel.patient.nunique(), "pacientes")
sel[["patient", "file", "montage", "fs", "edf_dur_s", "n_seiz_events_bi", "seiz_secs_bi"]]

## 9.2 Función de extracción para **un** fichero
Devuelve las ventanas y su metadata, añadiendo paciente y fichero (imprescindibles para la validación).

In [ ]:
WIN, STEP, SFREQ = 4.0, 2.0, 200.0

def extract(file_id, patient):
    raw = preprocess(read_tusz_raw(file_id), sfreq=SFREQ)
    if len(raw.ch_names) != 22:
        raise ValueError(f"{file_id}: {len(raw.ch_names)} canales")
    ep = make_windows(raw, win=WIN, step=STEP)
    ep.drop_bad(reject=dict(eeg=1500e-6))          # solo basura extrema (ver módulo 6)
    md = ep.metadata.assign(patient=patient, file=file_id)
    return ep.get_data(units="uV").astype(np.float32), md

X0, md0 = extract("aaaaabqv_s001_t000", "aaaaabqv")
print(X0.shape, md0.label.value_counts().to_dict())

## 9.3 Recorrer todos los ficheros (con caché)
Preprocesar es lo lento; guardamos cada fichero en `.npz` y si ya existe lo leemos. Así puedes
relanzar el notebook sin esperar. Si algún fichero falla, se registra y se sigue.

In [ ]:
CACHE = (SALIDAS / "cache_absz"); CACHE.mkdir(parents=True, exist_ok=True)
Xs, mds, errores = [], [], []
t0 = time.perf_counter()
for _, r in sel.iterrows():
    f = CACHE / f"{r.file}.npz"
    try:
        if f.exists():
            z = np.load(f, allow_pickle=True)
            X, md = z["X"], pd.DataFrame(z["md"].tolist(), columns=z["cols"])
        else:
            X, md = extract(r.file, r.patient)
            np.savez_compressed(f, X=X, md=md.to_records(index=False), cols=md.columns.values)
        Xs.append(X); mds.append(md)
    except Exception as e:                      # fichero no descargado, canales raros…
        errores.append((r.file, str(e)[:80]))
print(f"{len(Xs)} ficheros en {time.perf_counter() - t0:.0f} s | errores: {len(errores)}")
errores[:5]

In [ ]:
X = np.concatenate(Xs)
meta = pd.concat(mds, ignore_index=True)
y = (meta.label == "ictal").astype(int).values
groups = meta.patient.values
print("X:", X.shape, f"({X.nbytes / 1e9:.2f} GB)")
print("Proporción ictal:", y.mean().round(4))
meta.groupby("patient").label.value_counts().unstack(fill_value=0)

## 9.4 Baseline: potencia por bandas + regresión logística

**Regla de oro de la validación en EEG: separar por paciente.** Si ventanas del mismo paciente
caen en train y test, el modelo aprende al paciente (no la crisis) y las métricas se inflan.
TUSZ ya separa pacientes entre *train/dev/eval*; dentro de *train*, usa `GroupKFold`.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score, classification_report

# features: band power relativa en log, calculada con MNE sobre un EpochsArray
info = mne.create_info(TCP_NAMES, SFREQ, "eeg")
feats = band_power(mne.EpochsArray(X * 1e-6, info, verbose="error"), relative=True, log=True)
F = feats.reshape(len(feats), -1)               # (n, 22*5)
print("Features:", F.shape)

clf = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000))
cv = GroupKFold(n_splits=min(5, len(np.unique(groups))))
proba = cross_val_predict(clf, F, y, groups=groups, cv=cv, method="predict_proba")[:, 1]

print(f"ROC-AUC: {roc_auc_score(y, proba):.3f} | PR-AUC: {average_precision_score(y, proba):.3f} "
      f"(azar = {y.mean():.3f}) | Balanced acc: {balanced_accuracy_score(y, proba > 0.5):.3f}")
print(classification_report(y, proba > 0.5, target_names=["no crisis", "crisis"], digits=3))

> Con desbalanceo fuerte, **mira PR-AUC** (y sensibilidad / falsos positivos por hora), no la accuracy.
> En la literatura de detección en TUSZ se reportan también métricas por **evento** (¿se detectó la crisis?)
> además de por ventana.

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(y, proba, ax=axs[0], name="LogReg band power")
PrecisionRecallDisplay.from_predictions(y, proba, ax=axs[1], name="LogReg band power")
axs[0].plot([0, 1], [0, 1], "k--", lw=0.8); axs[1].axhline(y.mean(), c="k", ls="--", lw=0.8)
plt.tight_layout(); plt.show()

### Predicciones en el tiempo (vista "clínica")

In [ ]:
fid = "aaaaabqv_s001_t000"
m = meta.file == fid
plt.figure(figsize=(12, 2.5))
plt.plot(meta.start[m], proba[m], lw=0.8, label="P(crisis)")
plt.fill_between(meta.start[m], 0, y[m], step="post", color="tab:red", alpha=0.25, label="crisis real")
plt.ylim(0, 1); plt.xlabel("s"); plt.legend(); plt.title(f"{fid} · predicción fuera de muestra (GroupKFold)")
plt.show()

## 9.5 Herramientas de `mne.decoding`
MNE trae transformadores compatibles con sklearn que trabajan directamente con arrays `(n, canales, tiempo)`:
- `Scaler`: normaliza por canal.
- `Vectorizer`: aplana a `(n, canales*tiempo)`.
- `PSDEstimator`, `FilterEstimator`, `CSP`, `SlidingEstimator`…

In [ ]:
from mne.decoding import Scaler, Vectorizer, PSDEstimator
pipe = make_pipeline(PSDEstimator(sfreq=SFREQ, fmin=0.5, fmax=40),
                     Vectorizer(), StandardScaler(),
                     LogisticRegression(class_weight="balanced", max_iter=2000))
p2 = cross_val_predict(pipe, X.astype(np.float64), y, groups=groups, cv=cv, method="predict_proba")[:, 1]
print(f"PSD multitaper completo → ROC-AUC {roc_auc_score(y, p2):.3f} | PR-AUC {average_precision_score(y, p2):.3f}")

## 9.6 Exportar para PyTorch

Guarda un único fichero por *split* con la señal y las etiquetas. Formato recomendado: `.npy`
(se puede abrir con `mmap_mode="r"` para no cargar todo en RAM cuando crezca).

In [ ]:
OUT = (SALIDAS / "dataset_absz_train"); OUT.mkdir(parents=True, exist_ok=True)
np.save(OUT / "X.npy", X)                                       # (n, 22, 800) float32, µV
np.save(OUT / "y.npy", y)
meta.to_csv(OUT / "meta.csv", index=False)
print("Guardado en", OUT.resolve())

Y así lo leerías desde PyTorch (celda de referencia; no hace falta tener torch para el curso):

```python
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader

class TUSZWindows(Dataset):
    def __init__(self, folder):
        self.X = np.load(f"{folder}/X.npy", mmap_mode="r")
        self.y = np.load(f"{folder}/y.npy")
        self.meta = pd.read_csv(f"{folder}/meta.csv")
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        x = torch.from_numpy(np.array(self.X[i]))
        x = (x - x.mean(-1, keepdim=True)) / (x.std(-1, keepdim=True) + 1e-6)   # z-score por canal
        return x, torch.tensor(self.y[i], dtype=torch.float32)

ds = TUSZWindows(SALIDAS / "dataset_absz_train")
dl = DataLoader(ds, batch_size=64, shuffle=True)
xb, yb = next(iter(dl))        # xb: (64, 22, 800)  → entrada de una CNN 1D / EEGNet
```

---
## 🧪 Ejercicios finales

**E9.1** Sustituye la regresión logística por un `RandomForestClassifier(class_weight="balanced")`.
¿Mejora? Mira `feature_importances_` reorganizado como (22 canales × 5 bandas): ¿qué banda y región pesan más?
(Es tu primer contacto con explicabilidad.)

**E9.2** Calcula una métrica **por evento**: una crisis cuenta como detectada si alguna ventana con
`ictal_frac > 0` de esa crisis tiene `P > 0.5`. ¿Qué sensibilidad por evento obtienes?
¿Cuántos falsos positivos por hora?

**E9.3** Cambia `sel` para usar los ficheros `absz` de **dev** como test externo: entrena con todo train
y evalúa en dev. Así es como reportarás resultados en la memoria.

### Solución E9.1

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=0)
p_rf = cross_val_predict(rf, F, y, groups=groups, cv=cv, method="predict_proba")[:, 1]
print(f"RF → ROC-AUC {roc_auc_score(y, p_rf):.3f} | PR-AUC {average_precision_score(y, p_rf):.3f}")

rf.fit(F, y)
imp = pd.DataFrame(rf.feature_importances_.reshape(22, 5), index=TCP_NAMES, columns=list(BANDS))
fig, ax = plt.subplots(figsize=(5, 7))
im = ax.imshow(imp.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(5), imp.columns); ax.set_yticks(range(22), imp.index)
fig.colorbar(im, label="importancia"); ax.set_title("Importancia canal × banda (RF)"); plt.show()

### Solución E9.2

In [ ]:
# identificar cada crisis: fichero + intervalo; una ventana ictal pertenece a la crisis con la que solapa
hits, n_events = 0, 0
for fid_, g in meta.assign(p=proba).groupby("file"):
    raw_ann = read_tusz_raw(fid_, preload=False).annotations
    for a in raw_ann:
        s0, s1 = a["onset"], a["onset"] + a["duration"]
        w = g[(g.start < s1) & (g.start + WIN > s0)]
        n_events += 1
        hits += int((w.p > 0.5).any())
horas = meta.groupby("file").start.max().sum() / 3600
fp = ((proba > 0.5) & (y == 0)).sum()
print(f"Sensibilidad por evento: {hits}/{n_events} = {hits / n_events:.2f}")
print(f"Falsos positivos (ventanas): {fp} → {fp / horas:.1f} FP/hora (sin fusionar ventanas contiguas)")

### Solución E9.3 (esqueleto)

```python
sel_dev = files[(files.split == "dev") & files.labels.fillna("").str.contains("absz")
                & files.montage.isin(["01_tcp_ar", "02_tcp_le"])]
Xd, mdd = zip(*[extract(r.file, r.patient) for _, r in sel_dev.iterrows()])
Xd = np.concatenate(Xd); yd = (pd.concat(mdd).label == "ictal").astype(int).values
Fd = band_power(mne.EpochsArray(Xd * 1e-6, info), relative=True, log=True).reshape(len(Xd), -1)
clf.fit(F, y)
pd_ = clf.predict_proba(Fd)[:, 1]
print(roc_auc_score(yd, pd_), average_precision_score(yd, pd_))
```

---
## 🎓 ¿Y ahora qué?
Con esto tienes todo lo necesario de MNE para las fases 1-2 del TFM. Próximos pasos naturales:
1. Ampliar `sel` a todos los tipos de crisis y a *dev* → dataset de detección completo.
2. Encadenar los `t00x` de cada sesión para tener más contexto **preictal** (predicción).
3. Sustituir el baseline por una red (CNN 1D / EEGNet) leyendo `X.npy` con el `Dataset` de arriba.